# Data Cleaning

### Setup
Load the raw dataset and create a working copy. The original `insurance` DataFrame stays unchanged so it can be compared with the cleaned version.

In [ ]:
import pandas as pd
import numpy as np

insurance = pd.read_csv('insurance.csv')
insurance_clean = insurance.copy()

## 2.1 Overview
After completing the initial data exploration, several data quality issues were identified in the insurance dataset. These include missing values, inconsistent categorical values, invalid numerical values, and inconsistent formatting in the `charges` column.

The purpose of this stage is to clean and prepare the dataset so that it can be used for regression modelling.

## 2.2 Data Cleaning Objectives
The following issues will be addressed:

| Column | Identified Issue | Cleaning Approach |
|---|---|---|
| `age` | Missing and negative values | Handle missing and invalid values |
| `sex` | Inconsistent labels such as Male, Man and M | Standardize categories |
| `bmi` | Missing values | Handle missing values |
| `children` | Missing and negative values | Handle missing and invalid values |
| `smoker` | Missing values | Handle missing values |
| `region` | Inconsistent capitalization | Standardize text |
| `charges` | Currency symbols and missing values | Convert to numeric and remove records with missing targets |

## 2.3 Clean the Target Column: `charges`
The `charges` column is the target variable that the regression model will predict. Therefore, it must be converted into a numerical format before model training.

### 2.3.1 Remove Currency Symbols and Convert to Numeric
Some values contain a `$` symbol, while others are stored as numeric-looking strings. The column also contains invalid values such as `$nan`.

The values were converted into numeric format, with invalid entries treated as missing values.

* `.str.replace('$', '', regex=False)` removes the `$` symbol without treating it as a regular expression.
* `pd.to_numeric()` converts the cleaned strings into numbers.
* `errors='coerce'` converts values that cannot be interpreted as numbers into `NaN`.

In [ ]:
insurance_clean['charges'] = insurance_clean['charges'].astype(str).str.replace('$', '', regex=False)

insurance_clean['charges'] = pd.to_numeric(
    insurance_clean['charges'],
    errors='coerce'
)

print("Data type:")
print(insurance_clean['charges'].dtype)

print("\nMissing values:")
print(insurance_clean['charges'].isnull().sum())

print("\nSummary statistics:")
print(insurance_clean['charges'].describe())

#### Data type
**Interpretation:** Successfully converted `charges` from `object` to `float64`. The column can now be used for numerical calculations and regression modelling.

#### Missing values
**Interpretation:** There are now 66 missing values in `charges`, compared with the original 54.

> **Why did the count increase?**
> Some values, such as `$nan`, were previously stored as strings. After conversion, pandas recognized them as missing values. This is expected. We should remove rows with missing target values before training the model.

#### Summary statistics
| Statistic | Value | Interpretation |
|---|---:|---|
| Count | 1,272 | Number of records with valid charges |
| Mean | 13,286.59 | Average insurance charge |
| Standard deviation | 12,142.51 | Charges vary considerably |
| Minimum | 1,121.87 | Lowest observed charge |
| 25th percentile | 4,733.58 | 25% of valid charges are at or below this value |
| Median | 9,382.03 | Half of valid charges are at or below this value |
| 75th percentile | 16,579.96 | 75% of valid charges are at or below this value |
| Maximum | 63,770.43 | Highest observed charge |

**Key finding:** The mean is higher than the median, suggesting that the distribution of insurance charges is right-skewed. Some individuals have substantially higher charges than others.

The minimum charge is positive, and the maximum is plausible for an insurance dataset. There are no obvious problems in these summary statistics, although we should investigate unusual values further if needed.

### 2.3.2 Remove Rows with Missing Charges
Since `charges` is our **target variable**, we need to remove records where the charge amount is missing. A regression model cannot learn from a training record without a known target value.

#### Step 1: Check the number of missing values

In [ ]:
insurance_clean['charges'].isnull().sum()

#### Step 2: Remove rows with missing charges
Use `dropna()` with `subset=['charges']` so that only rows with missing target values are removed.
* `dropna()` removes rows containing missing values.
* `subset=['charges']` restricts the removal to rows where `charges` is missing.
* Other columns may still contain missing values. We will handle those separately.

In [ ]:
insurance_clean = insurance_clean.dropna(subset=['charges'])
print("Dataset shape:", insurance_clean.shape)
print("Missing charges:", insurance_clean['charges'].isnull().sum())

#### Interpretation
After converting the `charges` column into numeric format, 66 missing values were identified. Since `charges` is the target variable for the regression model, records without a valid target value cannot be used for supervised model training.

Therefore, rows with missing `charges` were removed using `dropna()` with the `charges` column specified as the subset.

**Result:**
* Original dataset: 1,338 rows
* Rows removed: 66
* Remaining dataset: 1,272 rows
* Missing values in `charges`: 0

Missing values in the other feature columns were retained for further cleaning and will be handled separately.

## 2.4 Handle Invalid Values in `age`
During exploration, we discovered that the `age` column contains negative values, such as -31, -27 and -64. These are invalid because a person's age cannot be negative.

### 2.4.1 Identify Negative Age Values
What we want to understand:
* How many records contain negative ages?
* What is the range of these invalid values?
* Are these records also missing other feature values?

#### Step 1: Count negative values

In [ ]:
(insurance_clean['age'] < 0).sum()

There are 63 records with negative ages out of the remaining 1,272 records. This means approximately 4.95% of the dataset contains invalid age values.

#### Step 2: Inspect the affected records

In [ ]:
insurance_clean[insurance_clean['age'] < 0]

Some records with negative ages also contain missing values in other columns, such as `bmi`, `sex`, `smoker`, `region` and `children`.
This means we need to consider both invalid values and missing values when preparing the dataset.

#### Step 3: Examine the distribution of negative ages

In [ ]:
insurance_clean.loc[insurance_clean['age'] < 0, 'age'].value_counts().sort_index()

The negative ages range from -18 to -64.
These values are not realistic because a person's age cannot be negative. Interestingly, their absolute values fall within a typical adult age range, but we cannot assume that the negative sign is simply a data-entry mistake that should be reversed.

### 2.4.2 Remove the Records with Negative Ages
We don't have enough information to confirm that -31 actually means 31. Converting them automatically could introduce incorrect information into our training dataset.

In [ ]:
insurance_clean = insurance_clean[(insurance_clean['age'] >= 0) | (insurance_clean['age'].isnull())]
print("Dataset shape:", insurance_clean.shape)
print("Negative ages remaining:", (insurance_clean['age'] < 0).sum())

### Interpretation
During data exploration, 63 records containing negative age values were identified. The invalid ages ranged from -18 to -64, accounting for approximately 4.95% of the dataset after removing records with missing `charges`.

Some affected records also contained missing values in other feature columns, including `sex`, `bmi`, `children`, `smoker` and `region`.

Since negative ages are not realistic and there is insufficient evidence to confirm that these values were caused by a reversed sign, they were treated as invalid records.

Records with negative ages were removed from the working dataset to prevent invalid age information from affecting the regression model.

**Result:**
* Records before removal: 1,272
* Records removed: 63
* Remaining records: 1,209
* Negative age values remaining: 0

Records with missing age values were retained for subsequent imputation rather than removed during invalid-value filtering.

## 2.5 Handle Invalid Values in `children`
During our earlier data exploration, we found that the `children` column contains negative values. Since the number of children cannot be negative, these values are invalid and need further investigation.

### 2.5.1 Identify Negative Children Values
#### Step 1: Count negative values

In [ ]:
(insurance_clean['children'] < 0).sum()

#### Step 2: Inspect affected records

In [ ]:
insurance_clean[insurance_clean['children'] < 0]

#### Step 3: Check the distribution of negative values

In [ ]:
insurance_clean.loc[insurance_clean['children'] < 0, 'children'].value_counts().sort_index()

From the affected records, we can observe:
1. Negative values range from -1 to -4.
2. The affected records contain a mixture of ages, genders, smoking statuses and regions.
3. There is no obvious pattern indicating that the negative values should simply be converted into positive values.
4. The actual number of children cannot be determined from the other columns.

### 2.5.2 Remove Records with Negative Children Values
We should not convert -1 to 1, for example, because we have no evidence that the negative sign is simply a data-entry mistake.

In [ ]:
insurance_clean = insurance_clean[(insurance_clean['children'] >= 0) | (insurance_clean['children'].isnull())]
print("Dataset shape:", insurance_clean.shape)
print("Negative children values:", (insurance_clean['children'] < 0).sum())
print("Missing children values:", insurance_clean['children'].isnull().sum())

### Interpretation
During data exploration, 41 records containing negative values in the `children` column were identified. The invalid values ranged from -1 to -4.

The affected records contained a mixture of ages, genders, smoking statuses and regions. However, there was no clear pattern to determine the actual number of children. Therefore, these values were treated as invalid rather than converted into positive values.

Records with negative `children` values were removed from the working dataset to prevent invalid information from affecting the regression model. Missing values were retained for further handling during the missing value treatment stage.

**Result:**
* Records before removal: 1,209
* Records removed: 41
* Remaining records: 1,168
* Negative children values remaining: 0
* Missing children values retained: 29

## 2.6 Standardize Categorical Values
During our earlier data exploration, we found inconsistent values in the categorical columns. For example, Male, male, M and man represent the same category but are recorded differently. These inconsistencies need to be standardized so the model can interpret the categories correctly.

### 2.6.1 Sex
#### Step 1: Inspect current values

In [ ]:
insurance_clean['sex'].value_counts(dropna=False)

#### Step 2: Inspect unique values

In [ ]:
insurance_clean['sex'].unique()

### 2.6.2 Standardize the `sex` column

In [ ]:
sex_mapping = {
    'male': 'male',
    'M': 'male',
    'man': 'male',
    'female': 'female',
    'F': 'female',
    'woman': 'female'
}

insurance_clean['sex'] = insurance_clean['sex'].map(sex_mapping)
insurance_clean['sex'].value_counts(dropna=False)

### 2.6.3 Smoker
#### Step 1: Check value counts

In [ ]:
insurance_clean['smoker'].value_counts(dropna=False)

#### Step 2: Check unique values

In [ ]:
insurance_clean['smoker'].unique()

> The column contains only two valid categories. There are no inconsistent variations such as Yes, YES or No. Therefore, no standardization is required for this column.

### 2.6.4 Region
#### Step 1: Check value counts

In [ ]:
insurance_clean['region'].value_counts(dropna=False)

#### Step 2: Check unique values

In [ ]:
insurance_clean['region'].unique()

### 2.6.5 Standardize the `region` column
`.str.title()` converts the first letter of each region to uppercase while keeping the remaining letters lowercase. Missing values remain unchanged.

In [ ]:
insurance_clean['region'] = insurance_clean['region'].str.title()
insurance_clean['region'].value_counts(dropna=False)

### Interpretation
During data exploration, inconsistent categorical values were identified in the `sex` and `region` columns. The `sex` column contained different representations of male and female, while the `region` column contained inconsistent capitalization for the four regions.

The `.map()` method was used with a mapping dictionary to standardize the `sex` column into two categories, male and female. Meanwhile, the `.str.title()` method was applied to the `region` column to ensure consistent capitalization.

The `smoker` column was also inspected and found to contain only two consistent categories, yes and no. Therefore, no changes were required.

Missing values in all three columns were preserved for further handling during the missing value treatment stage.

**Result:**
* **sex**: 582 male, 557 female, 29 missing values.
* **smoker**: 904 no, 233 yes, 31 missing values.
* **region**: 304 Southeast, 284 Northwest, 277 Northeast, 272 Southwest, 31 missing values.

All categorical values are now consistently represented, making the dataset more suitable for further preprocessing and regression modelling.

## 2.7 Handle Missing Feature Values
So far, we have removed records with missing `charges` because `charges` is our target variable. However, we have retained missing values in the feature columns because we may still be able to use those records for model training.

The next step is to understand how many missing values remain and decide how to handle them.

### 2.7.1 Identify Missing Values
#### Step 1: Check missing value counts
This shows the number of missing values in each column.

In [ ]:
insurance_clean.isnull().sum()

#### Step 2: Calculate missing value percentages
This helps us understand the proportion of missing values in each column.

In [ ]:
insurance_clean.isnull().sum() / len(insurance_clean) * 100

> Since the missing percentages are relatively low, we can consider imputation instead of removing more records. However, we should not automatically fill every missing value with the mean or mode. We need to choose a suitable method for each column.

### 2.7.2 Determine the Appropriate Imputation Method
Before filling the missing values, let's inspect the distribution of each feature to decide which imputation method is appropriate.

#### Step 1: Inspect numerical columns
We'll use the results to decide whether to use the mean or median.

| Column | Possible treatment | Reason |
|---|---|---|
| `age` | Median | Less affected by extreme values |
| `bmi` | Median | More robust if the distribution is skewed |
| `children` | Median | Preserves a typical whole-number value |

In [ ]:
insurance_clean[['age', 'bmi', 'children']].describe()

#### Step 2: Inspect categorical columns
We'll use the mode (most frequent category) as a possible treatment for missing categorical values.

In [ ]:
for column in ['sex', 'smoker', 'region']:
    print(f"\n{column.upper()}")
    print(insurance_clean[column].value_counts())

### 2.7.3 Selected Imputation Methods
Based on the findings, we will use the following methods:

| Column | Data type | Imputation method | Replacement value |
|---|---|---|---|
| `age` | Numerical | Median | 39 |
| `bmi` | Numerical | Median | 30.21 |
| `children` | Numerical | Median | 1 |
| `sex` | Categorical | Mode | male |
| `smoker` | Categorical | Mode | no |
| `region` | Categorical | Mode | Southeast |

### Interpretation
The dataset contains missing values across all six feature columns, with missing percentages ranging from approximately 2.48% to 3.17%.

For numerical columns (`age`, `bmi` and `children`), median imputation was selected because the median is less sensitive to extreme values. For categorical columns (`sex`, `smoker` and `region`), mode imputation was selected to replace missing values with the most frequently occurring category.

The missing values will be handled during model preprocessing using statistics calculated from the training data only to prevent data leakage.

#### Important: Should we fill the missing values now?
Not yet. Since we will evaluate our regression model using training and testing data, we should calculate imputation values using the training data only.

Otherwise, information from the testing data could influence the training process. This is known as **data leakage**.

Later, we will use Scikit-learn's `SimpleImputer` to handle missing values as part of our model preprocessing pipeline.

## 2.8 Final Data Quality Verification
### 2.8.1 Check Dataset Structure
#### Step 1: Check the dataset shape
This tells us how many records and columns remain after cleaning.

In [ ]:
print("Dataset shape:", insurance_clean.shape)

#### Step 2: Check data types and non-null counts
This helps us verify that:
* `age`, `bmi`, `children` and `charges` have numerical data types.
* `sex`, `smoker` and `region` have appropriate categorical data types.
* `charges` contains no missing values.

In [ ]:
insurance_clean.info()

### 2.8.2 Verify Remaining Invalid Values
#### Step 1: Check negative values

In [ ]:
print("Negative ages:", (insurance_clean['age'] < 0).sum())
print("Negative children:", (insurance_clean['children'] < 0).sum())

#### Step 2: Check categorical values

In [ ]:
for column in ['sex', 'smoker', 'region']:
    print(f"\n{column.upper()}")
    print(insurance_clean[column].value_counts(dropna=False))

### 2.8.3 Final Dataset Summary

In [ ]:
print("Final dataset shape:", insurance_clean.shape)

print("\nTotal missing values:")
print(insurance_clean.isnull().sum().sum())

print("\nDuplicate rows:")
print(insurance_clean.duplicated().sum())

print("\nData types:")
print(insurance_clean.dtypes)

### 2.8.4 Investigate Duplicate Records
#### Step 1: Display duplicate records
The `keep=False` argument displays all records involved in duplication, rather than showing only the later occurrence.

In [ ]:
insurance_clean[insurance_clean.duplicated(keep=False)]

#### Step 2: Check the duplicate count
We need to determine whether the records are exact duplicates across all seven columns.

In [ ]:
insurance_clean.duplicated().sum()

### 2.8.5 Remove Duplicate Records

In [ ]:
insurance_clean = insurance_clean.drop_duplicates()
print("Dataset shape:", insurance_clean.shape)
print("Duplicate rows:", insurance_clean.duplicated().sum())

### Interpretation
During the final data quality verification, one pair of exact duplicate records was identified (indices 195 and 581). Both records contained identical values across all seven columns. Therefore, the `drop_duplicates()` method was applied to remove one duplicate record and prevent the same observation from being counted twice during model training. After removal, the dataset contained 1,167 records with no remaining duplicate rows.